In [7]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [10]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [11]:
MAX_LENGTH = 10

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [12]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counting words...
Counted words:
fra 4601
eng 2991
['c est mon camarade de classe', 'he is my classmate']


In [13]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

In [14]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

In [15]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [16]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [17]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [18]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [19]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [20]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [21]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [22]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [23]:
hidden_size = 128
batch_size = 32

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counting words...
Counted words:
fra 4601
eng 2991
0m 45s (- 11m 21s) (5 6%) 1.5492
1m 25s (- 9m 56s) (10 12%) 0.6917
2m 4s (- 8m 57s) (15 18%) 0.3654
2m 42s (- 8m 8s) (20 25%) 0.2059
3m 21s (- 7m 24s) (25 31%) 0.1286
4m 1s (- 6m 41s) (30 37%) 0.0894
4m 40s (- 6m 0s) (35 43%) 0.0685
5m 19s (- 5m 19s) (40 50%) 0.0558
5m 58s (- 4m 38s) (45 56%) 0.0476
6m 37s (- 3m 58s) (50 62%) 0.0420
7m 16s (- 3m 18s) (55 68%) 0.0388
7m 56s (- 2m 38s) (60 75%) 0.0358
8m 35s (- 1m 58s) (65 81%) 0.0337
9m 14s (- 1m 19s) (70 87%) 0.0317
9m 53s (- 0m 39s) (75 93%) 0.0312
10m 33s (- 0m 0s) (80 100%) 0.0297


In [24]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis dur a cuire
= i m tough
< i m tough <EOS>

> tu es tout a fait attirant
= you re quite attractive
< you re quite attractive <EOS>

> nous ne faisons que parler
= we re just talking
< we re just talking talking <EOS>

> elle manque de sens commun
= she is lacking in common sense
< she is lacking in common sense <EOS>

> je suis desole pour ce que j ai fait
= i m sorry for what i have done
< i m sorry for what i did <EOS>

> elle est plus maligne que lui
= she s smarter than him
< she s smarter than him <EOS>

> vous vous foutez de moi
= you re putting me on
< you re putting me on <EOS>

> je vais etudier le francais l annee prochaine
= i m going to study french next year
< i m going to study french next year <EOS>

> je suis americaine
= i am american
< i am american <EOS>

> je ne suis pas d humeur pour le moment
= i m not in the mood right now
< i m not in the mood right now <EOS>



In [26]:
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap='bone')
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticklabels([''] + input_sentence.split(' ') +
                       ['<EOS>'], rotation=90)
    ax.set_yticklabels([''] + output_words)

    # Show label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()


def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(encoder, decoder, input_sentence, input_lang, output_lang)
    print('input =', input_sentence)
    print('output =', ' '.join(output_words))
    # showAttention(input_sentence, output_words, attentions[0, :len(output_words), :])


evaluateAndShowAttention('il n est pas aussi grand que son pere')

evaluateAndShowAttention('je suis trop fatigue pour conduire')

evaluateAndShowAttention('je suis desole si c est une question idiote')

evaluateAndShowAttention('je suis reellement fiere de vous')

input = il n est pas aussi grand que son pere
output = he is not as tall as his father <EOS>
input = je suis trop fatigue pour conduire
output = i m too tired to drive her safety <EOS>
input = je suis desole si c est une question idiote
output = i m sorry if this is a stupid question <EOS>
input = je suis reellement fiere de vous
output = i m really proud of you <EOS>


In [ ]:
# Saving checkpoint
bundle = {
    "encoder": encoder.state_dict(),
    "decoder": decoder.state_dict(),
    "config": {
        "hidden_size": hidden_size,
        "n_layers": 1,
        "dropout": 0.1,
        "max_length": MAX_LENGTH,
        "SOS_token": SOS_token,
        "EOS_token": EOS_token,
    },

    "input_lang": {
        "word2index": getattr(input_lang, "word2index", {}),
        "index2word": getattr(input_lang, "index2word", {}),
        "n_words":    getattr(input_lang, "n_words", len(getattr(input_lang, "word2index", {})))
    },
    "output_lang": {
        "word2index": getattr(output_lang, "word2index", {}),
        "index2word": getattr(output_lang, "index2word", {}),
        "n_words":    getattr(output_lang, "n_words", len(getattr(output_lang, "word2index", {})))
    }
}

torch.save(bundle, "checkpoint.pt")
print("Saved checkpoint.pt")


#Inference

In [6]:
import torch, torch.nn as nn, torch.nn.functional as F
from typing import Dict, Any, Tuple

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def infer_gru_sizes(gru_weight_ih, gru_weight_hh) -> Tuple[int, int]:
    H = gru_weight_hh.shape[1]
    in_dim = gru_weight_ih.shape[1]
    return in_dim, H

def peek_checkpoint(ckpt_path: str):
    bundle = torch.load(ckpt_path, map_location=DEVICE)
    enc_sd = bundle.get("encoder") or bundle.get("encoder_state_dict")
    dec_sd = bundle.get("decoder") or bundle.get("decoder_state_dict")
    assert isinstance(enc_sd, dict) and isinstance(dec_sd, dict), "checkpoint 需要 encoder/decoder 權重"
    e_in, e_H = infer_gru_sizes(enc_sd["gru.weight_ih_l0"], enc_sd["gru.weight_hh_l0"])
    d_in, d_H = infer_gru_sizes(dec_sd["gru.weight_ih_l0"], dec_sd["gru.weight_hh_l0"])

    if "embedding.weight" in dec_sd:
        dec_embed_dim = dec_sd["embedding.weight"].shape[1]
    else:
        dec_embed_dim = d_in

    has_additive = any(k.startswith("attention.Wa.") for k in dec_sd.keys())
    has_attn_combine = ("attn_combine.weight" in dec_sd)

    return bundle, (e_in, e_H), (d_in, d_H), dec_embed_dim, has_additive, has_attn_combine

# Model
class EncoderRNNFlex(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int, n_layers: int = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_size, num_layers=n_layers)

    def forward(self, token, hidden):
        emb = self.embedding(token).view(1,1,-1)
        out, hidden = self.gru(emb, hidden)
        return out, hidden

    def initHidden(self, batch_size=1):
        return torch.zeros(self.n_layers, batch_size, self.hidden_size, device=DEVICE)

# Bahdanau additive
class AdditiveAttention(nn.Module):
    def __init__(self, embed_dim: int, hidden_size: int):
        super().__init__()
        self.Wa = nn.Linear(embed_dim, hidden_size, bias=True)
        self.Ua = nn.Linear(hidden_size, hidden_size, bias=True)
        self.Va = nn.Linear(hidden_size, 1, bias=True)

    def forward(self, embedded, hidden, enc_outs):
        e = embedded[0]
        S = enc_outs.size(0)
        e_exp = self.Wa(e).expand(S, -1)
        h_exp = self.Ua(hidden[0]).expand(S, -1)
        score = self.Va(torch.tanh(e_exp + h_exp)).squeeze(1)
        attn_w = F.softmax(score, dim=0).unsqueeze(0)
        return attn_w

class AttnDecoderAdditiveConcatGRU(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, embed_dim: int, n_layers: int = 1, dropout_p: float = 0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = AdditiveAttention(embed_dim, hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_dim + hidden_size, hidden_size, num_layers=n_layers)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, token, hidden, enc_outs):
        emb = self.embedding(token).view(1,1,-1)
        emb = self.dropout(emb)
        attn_w = self.attention(emb, hidden, enc_outs)
        ctx = torch.bmm(attn_w.unsqueeze(0), enc_outs.unsqueeze(0))
        x = torch.cat((emb, ctx), dim=2)
        out, hidden = self.gru(x, hidden)
        logits = F.log_softmax(self.out(out[0]), dim=1)
        return logits, hidden, attn_w

def build_models_from_ckpt(ckpt_path: str):
    bundle, (enc_in, enc_H), (dec_in, dec_H), dec_E, has_add, has_combine = peek_checkpoint(ckpt_path)
    cfg = bundle.get("config", {})
    n_layers = cfg.get("n_layers", 1)
    dropout = cfg.get("dropout", 0.1)
    max_length = cfg.get("max_length", 10)

    if "input_lang" in bundle and "output_lang" in bundle:
        inp = bundle["input_lang"]; outp = bundle["output_lang"]
        src_w2i, tgt_w2i = inp["word2index"], outp["word2index"]
        src_n = inp.get("n_words", len(src_w2i)); tgt_n = outp.get("n_words", len(tgt_w2i))
        idx2tgt = outp["index2word"]
    else:
        src_w2i = bundle["stoi_src"]; tgt_w2i = bundle["stoi_tgt"]
        itos_tgt = bundle.get("itos_tgt")
        src_n = len(bundle.get("itos_src", [])) or len(src_w2i)
        tgt_n = len(itos_tgt) if isinstance(itos_tgt, list) else len(tgt_w2i)
        idx2tgt = itos_tgt if isinstance(itos_tgt, list) else {int(k):v for k,v in bundle.get("itos_tgt", {}).items()}

    SOS = cfg.get("SOS_token", 0); EOS = cfg.get("EOS_token", 1)

    encoder = EncoderRNNFlex(src_n, enc_in, enc_H, n_layers=n_layers).to(DEVICE)

    assert has_add, "checkpoint is not additive attention（missing attention.Wa/Ua/Va weights）"
    assert not has_combine, "weights includes attn_combine；please use attn_combine"
    decoder = AttnDecoderAdditiveConcatGRU(dec_H, tgt_n, embed_dim=dec_E, n_layers=n_layers, dropout_p=dropout).to(DEVICE)

    enc_sd = bundle.get("encoder") or bundle.get("encoder_state_dict")
    dec_sd = bundle.get("decoder") or bundle.get("decoder_state_dict")
    encoder.load_state_dict(enc_sd, strict=True)
    decoder.load_state_dict(dec_sd, strict=True)

    meta = {"cfg": cfg, "src_vocab": src_w2i, "idx2tgt": idx2tgt, "SOS": SOS, "EOS": EOS, "max_length": max_length}
    return encoder, decoder, meta

@torch.no_grad()
def tensorFromSentence(word2index: Dict[str,int], sentence: str, EOS: int):
    toks = " ".join(sentence.strip().lower().split()).split()
    ids = [word2index.get(w, word2index.get("<unk>", 0)) for w in toks] + [EOS]
    return torch.tensor(ids, dtype=torch.long, device=DEVICE).view(-1,1)

@torch.no_grad()
def translate_once(encoder, decoder, text: str, meta: Dict[str,Any]) -> str:
    max_length = meta["max_length"]; EOS = meta["EOS"]; SOS = meta["SOS"]
    src_vocab = meta["src_vocab"]; idx2tgt = meta["idx2tgt"]

    inp = tensorFromSentence(src_vocab, text, EOS)
    hid = encoder.initHidden()
    enc_outs = torch.zeros(max_length, encoder.hidden_size, device=DEVICE)
    S = min(inp.size(0), max_length)
    for i in range(S):
        o, hid = encoder(inp[i], hid)
        enc_outs[i] = o[0,0]

    di = torch.tensor([[SOS]], device=DEVICE)
    hid_d = hid
    out_ids = []
    for _ in range(max_length):
        do, hid_d, _ = decoder(di, hid_d, enc_outs)
        nxt = do.topk(1)[1].item()
        if nxt == EOS: break
        out_ids.append(nxt)
        di = torch.tensor([[nxt]], device=DEVICE)

    if isinstance(idx2tgt, dict):
        toks = [idx2tgt.get(i, "<unk>") for i in out_ids]
    else:
        toks = [idx2tgt[i] if 0 <= i < len(idx2tgt) else "<unk>" for i in out_ids]
    return " ".join(toks)

# Insert checkpoint.pt and type French to translate to English

In [8]:
ckpt_path = "checkpoint.pt"
encoder, decoder, meta = build_models_from_ckpt(ckpt_path)
translation = lambda s: translate_once(encoder, decoder, s, meta)

print(translation("je suis desole si c est une question idiote"))

i m sure that s ll come this
